# LangChain.js RAG with Oracle AI Database

This notebook shows how to build a JavaScript RAG workflow with LangChain.js and Oracle AI Database.

It focuses on the TypeScript/JavaScript integration package `@oracle/langchain-oracledb`. The runnable notebook uses JavaScript modules so it can execute directly with Node.js, and the same imports can be used from TypeScript with a normal `tsconfig` build.

**What this notebook demonstrates**

- Connect from Node.js to FreeSQL, Autonomous AI Database, or local Oracle Database.
- Validate the schema before creating vector-store objects.
- Store documents and metadata with `OracleVS`.
- Inspect the Oracle table created by the vector store.
- Run similarity search, metadata-filtered search, and MMR retrieval.
- Show where vector index creation fits when the database environment has enough quota.
- Generate a grounded RAG answer with a chat model from retrieved OracleVS context.

## Architecture at a Glance

```text
                 +-------------------------------+
                 | JavaScript / TypeScript app   |
                 | Node.js + LangChain.js        |
                 +---------------+---------------+
                                 |
                                 | uses
                                 v
                 +-------------------------------+
                 | @oracle/langchain-oracledb    |
                 | OracleVS vector store         |
                 +---------------+---------------+
                                 |
                                 | stores and queries
                                 v
+----------------------------------------------------------------+
|                     Oracle AI Database                         |
|  VECTOR embeddings + document text + JSON metadata + indexes   |
+-------------------------------+--------------------------------+
                                |
                                | returns retrieved context
                                v
                 +-------------------------------+
                 | Grounded RAG response         |
                 +-------------------------------+
```

`OracleVS` is the main integration point. The application keeps the LangChain developer experience, while Oracle AI Database stores the text, embedding vectors, and metadata used for retrieval.

## What the Oracle LangChain.js Package Provides

The Oracle JavaScript integration package connects LangChain.js applications to Oracle AI Database. In the official guide, the package surface includes vector storage, similarity search, document loading, text splitting, in-database embeddings, summarization, and vector index creation.

This notebook focuses the runnable path on the pieces that are most useful for a first Developer Hub tutorial and work well on FreeSQL:

| Capability | Why it matters in a RAG app | Covered in runnable cells |
| --- | --- | --- |
| `OracleVS` | Stores text, metadata, and embeddings in Oracle AI Database | Yes |
| Similarity search | Retrieves nearest chunks for a user question | Yes |
| Metadata filtering | Restricts retrieval using JSON metadata predicates | Yes |
| MMR retrieval | Balances relevance and diversity in retrieved context | Yes |
| `createIndex` | Adds HNSW/IVF vector indexes when the DB environment supports it | Opt-in |
| `OracleDocLoader`, `OracleTextSplitter`, `OracleEmbeddings`, `OracleSummary` | Useful extensions for production ingestion and Oracle-backed model calls | Explained at the end |

## Database Connection Guide

Use one of these connection paths. The notebook reads the same `.env` variable names for all three options.

<details open>
<summary><strong>FreeSQL</strong></summary>

Use FreeSQL when you want a hosted Oracle Database schema for a tutorial without setting up a local database.

**Step 1: Open FreeSQL**

Open [FreeSQL](https://freesql.com). The worksheet interface is where you can browse schema objects, run SQL, and access the database connection details.

<p align="center">
  <img src="../images/freesql/freesql-interface.png" alt="FreeSQL worksheet interface" width="760">
</p>
<br>

**Step 2: Sign in**

Select **Sign In** and authenticate with your Oracle account, or create an Oracle account if you do not already have one.

<p align="center">
  <img src="../images/freesql/freesql-sign-in.png" alt="Oracle sign-in page for FreeSQL" width="560">
</p>
<br>

**Step 3: Copy the NodeJS connection details**

After sign-in, select **Connect to the Database** from the top navigation. Choose the **NodeJS** tab for this LangChain.js notebook, then copy the generated username, password, and connection string.

<p align="center">
  <img src="../images/freesql/freesql-nodejs-connection.png" alt="FreeSQL NodeJS connection details" width="760">
</p>
<br>

**Step 4: Create the notebook `.env` file**

Add the values to a private `.env` file in the same folder as this notebook.

```env
DB_USER=<freesql-user>
DB_PASSWORD=<freesql-password>
DB_CONNECT_STRING=<freesql-connect-descriptor-or-service-url>
LANGCHAIN_JS_TABLE=LC_JS_RAG_DEMO
LANGCHAIN_JS_CREATE_INDEX=false
```

FreeSQL normally works for this notebook path because it uses regular schema tables. Vector index creation is disabled by default because small hosted schemas may not have enough quota for index auxiliary objects.

</details>

<details>
<summary><strong>Autonomous AI Database</strong></summary>

Use Autonomous AI Database when you want the same LangChain.js workflow against a managed Oracle AI Database environment.

```env
DB_USER=<database-user>
DB_PASSWORD=<database-password>
DB_CONNECT_STRING=<adb-service-name-or-connect-descriptor>
LANGCHAIN_JS_TABLE=LC_JS_RAG_DEMO
LANGCHAIN_JS_CREATE_INDEX=true
```

</details>

<details>
<summary><strong>Local Oracle Database</strong></summary>

Use a local Oracle Database when you want a repeatable development environment on your machine or in a container.

```env
DB_USER=<local-user>
DB_PASSWORD=<local-password>
DB_CONNECT_STRING=localhost:1521/<service-name>
LANGCHAIN_JS_TABLE=LC_JS_RAG_DEMO
LANGCHAIN_JS_CREATE_INDEX=true
```

</details>


## 1. Prepare the JavaScript Workspace

In [1]:
from pathlib import Path
import json
import os
import subprocess
import tempfile

NOTEBOOK_DIR = Path.cwd()
RUNTIME_DIR = Path(tempfile.mkdtemp(prefix="langchain-js-oracle-"))
NPM_BIN = "npm.cmd" if os.name == "nt" else "npm"
RUN_ENV = os.environ.copy()
RUN_ENV["NOTEBOOK_ENV_PATH"] = str(NOTEBOOK_DIR / ".env")

print("JavaScript runtime ready.")

JavaScript runtime ready.


In [2]:
package_json = {
    "name": "langchain-js-oracle-ai-database-rag",
    "version": "1.0.0",
    "private": True,
    "type": "module",
    "scripts": {
        "demo": "node oracle_langchain_rag_demo.mjs"
    },
    "dependencies": {
        "@langchain/core": "^1.0.0",
        "@langchain/openai": "^1.0.0",
        "@oracle/langchain-oracledb": "^1.0.0",
        "dotenv": "^17.2.3",
        "oracledb": "^6.10.0"
    }
}

_ = (RUNTIME_DIR / "package.json").write_text(json.dumps(package_json, indent=2), encoding="utf-8")
print("Node.js package manifest ready.")

Node.js package manifest ready.


In [3]:
result = subprocess.run(
    [NPM_BIN, "install"],
    cwd=RUNTIME_DIR,
    env=RUN_ENV,
    text=True,
    encoding="utf-8",
    errors="replace",
    capture_output=True,
)

if result.returncode != 0:
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
    raise RuntimeError("npm install failed. Check the npm output above.")

print("Node.js dependencies installed.")

Node.js dependencies installed.


## 2. Configure the Connection Profile

Create a private `.env` file in the same folder as this notebook. The notebook uses these values from JavaScript through `dotenv`, but it never prints the credentials.

```env
DB_USER=<database-user>
DB_PASSWORD=<database-password>
DB_CONNECT_STRING=<database-connect-string>
LANGCHAIN_JS_TABLE=LC_JS_RAG_DEMO
LANGCHAIN_JS_CREATE_INDEX=false

# Required for the final RAG generation step.
# Use OPENAI_API_KEY, or keep MODEL_PROVIDER_API_KEY if your environment already uses that name.
OPENAI_API_KEY=<openai-api-key>
MODEL_PROVIDER_API_KEY=<optional-existing-provider-key>
LANGCHAIN_JS_LLM_MODEL=gpt-4o-mini
```

For FreeSQL, use **Connect to the Database** and choose the **NodeJS** tab for this LangChain.js notebook. Copy the generated username, password, and connection string into the private `.env` file. The next executable step validates the connection and prints only the database target, for example `Connected to FreeSQL`.

The final RAG section sends the retrieved OracleVS context to the configured chat model. If your private `.env` already has `MODEL_PROVIDER_API_KEY`, the JavaScript code will use it automatically; otherwise set `OPENAI_API_KEY`.


## 3. Create the Database Connection Module

This JavaScript module loads the private `.env` file, sets the notebook program identifier, and exposes a reusable `node-oracledb` pool factory. It is the only place that reads credentials.

In [4]:
db_config_js = r'''
import fs from "node:fs";
import path from "node:path";
import process from "node:process";

import dotenv from "dotenv";
import oracledb from "oracledb";

const runtimeDir = process.cwd();
dotenv.config({ path: process.env.NOTEBOOK_ENV_PATH || path.resolve(runtimeDir, "..", ".env") });

if (oracledb.defaults) {
  oracledb.defaults.program = "devrel-developerhub-langchain-js-oracle-rag";
}

export const tableName = (process.env.LANGCHAIN_JS_TABLE || "LC_JS_RAG_DEMO").toUpperCase();
export const createVectorIndex = (process.env.LANGCHAIN_JS_CREATE_INDEX || "false").toLowerCase() === "true";
export const resultsPath = path.join(runtimeDir, "rag_results.json");
export const llmApiKey = process.env.OPENAI_API_KEY || process.env.MODEL_PROVIDER_API_KEY;
export const llmModel = process.env.LANGCHAIN_JS_LLM_MODEL || process.env.OPENAI_MODEL || process.env.OAMP_LLM_MODEL || "gpt-4o-mini";

const dbUser = process.env.DB_USER || process.env.ORACLEDB_USER;
const dbPassword = process.env.DB_PASSWORD || process.env.ORACLEDB_PASSWORD;
const dbConnectString =
  process.env.DB_CONNECT_STRING ||
  process.env.DB_DSN ||
  process.env.ORACLEDB_CONNECTION_STRING;

const missing = [
  ["DB_USER", dbUser],
  ["DB_PASSWORD", dbPassword],
  ["DB_CONNECT_STRING or DB_DSN", dbConnectString],
]
  .filter(([, value]) => !value)
  .map(([name]) => name);

if (missing.length > 0) {
  throw new Error(`Missing required environment variables: ${missing.join(", ")}`);
}

export function writeResults(payload) {
  fs.writeFileSync(resultsPath, JSON.stringify(payload, null, 2));
}

export async function createPool() {
  return oracledb.createPool({
    user: dbUser,
    password: dbPassword,
    connectString: dbConnectString,
  });
}
'''

_ = (RUNTIME_DIR / "db_config.mjs").write_text(db_config_js, encoding="utf-8")
print("Database connection module ready.")


Database connection module ready.


## 4. Add Database Preflight Checks

Before using `OracleVS`, the notebook verifies that the schema can create, insert into, select from, and drop normal application tables. This keeps FreeSQL failures easy to diagnose.

In [5]:
db_preflight_js = r"""
export async function tableExists(connection, name) {
  const result = await connection.execute(
    `SELECT COUNT(*) FROM user_tables WHERE table_name = :name`,
    [name.toUpperCase()],
  );
  return Number(result.rows[0][0]) > 0;
}

export async function runTablePreflight(connection, summary) {
  const diagTable = "LC_JS_PREFLIGHT";
  try {
    await connection.execute(`DROP TABLE ${diagTable} PURGE`);
  } catch (error) {
    if (!String(error.message).includes("ORA-00942")) throw error;
  }

  await connection.execute(`CREATE TABLE ${diagTable} (id NUMBER PRIMARY KEY, note VARCHAR2(100))`);
  await connection.execute(`INSERT INTO ${diagTable} VALUES (1, 'table permission check')`, [], { autoCommit: true });
  const rows = await connection.execute(`SELECT COUNT(*) FROM ${diagTable}`);
  await connection.execute(`DROP TABLE ${diagTable} PURGE`);

  summary.push({
    step: "table_operations",
    status: "PASS",
    detail: `Created, inserted, selected ${rows.rows[0][0]} row, and dropped a test table`,
  });
}

export async function checkTablespaceQuota(connection) {
  const result = await connection.execute(
    `SELECT tablespace_name, bytes, max_bytes FROM user_ts_quotas WHERE tablespace_name = 'USERS'`,
  );

  if (!result.rows || result.rows.length === 0) {
    return { status: "UNKNOWN", detail: "No USERS quota row returned for this schema" };
  }

  const [tablespaceName, usedBytes, maxBytes] = result.rows[0];
  if (Number(maxBytes) < 0) {
    return { status: "PASS", detail: `${tablespaceName}: unlimited quota` };
  }

  const freeBytes = Number(maxBytes) - Number(usedBytes);
  const usedMb = Number(usedBytes) / 1024 / 1024;
  const maxMb = Number(maxBytes) / 1024 / 1024;
  const freeMb = freeBytes / 1024 / 1024;

  if (freeBytes < 2_500_000) {
    return {
      status: "MISSING",
      detail: `${tablespaceName}: ${usedMb.toFixed(2)} MB used of ${maxMb.toFixed(2)} MB; only ${freeMb.toFixed(2)} MB remains. Use a fresh FreeSQL schema or drop old demo objects before creating the OracleVS table.`,
    };
  }

  return {
    status: "PASS",
    detail: `${tablespaceName}: ${usedMb.toFixed(2)} MB used of ${maxMb.toFixed(2)} MB; ${freeMb.toFixed(2)} MB remains`,
  };
}
"""

_ = (RUNTIME_DIR / "db_preflight.mjs").write_text(db_preflight_js, encoding="utf-8")
print("Database preflight checks ready.")

Database preflight checks ready.


## 5. Add Vector Table Inspection

After `OracleVS` runs, this helper reads Oracle data dictionary views so the notebook can show what table, columns, and indexes were created.

In [6]:
db_inspection_js = r"""
export async function inspectVectorTable(connection, name) {
  const tableCount = await connection.execute(`SELECT COUNT(*) FROM ${name}`);
  const columnRows = await connection.execute(
    `SELECT column_name, data_type
     FROM user_tab_columns
     WHERE table_name = :tableName
     ORDER BY column_id`,
    [name],
  );
  const indexRows = await connection.execute(
    `SELECT index_name, index_type
     FROM user_indexes
     WHERE table_name = :tableName
     ORDER BY index_name`,
    [name],
  );
  const storageRows = await connection.execute(
    `SELECT ROUND(SUM(bytes)/1024/1024, 3) AS mb
     FROM user_segments
     WHERE segment_name = :tableName`,
    [name],
  );

  return {
    rowCount: Number(tableCount.rows[0][0]),
    columns: columnRows.rows.map(([column, type]) => ({ column, type })),
    indexes: indexRows.rows.map(([index, type]) => ({ index, type })),
    storageMb: Number(storageRows.rows[0][0] || 0),
  };
}
"""

_ = (RUNTIME_DIR / "db_inspection.mjs").write_text(db_inspection_js, encoding="utf-8")
print("Vector table inspection ready.")

Vector table inspection ready.


## 6. Test the Database Connection

Run a direct `node-oracledb` connection check before the LangChain.js workflow. This confirms that Node.js can open a database session with the `.env` values, without printing credentials.

In [7]:
connection_check_js = r"""
import { createPool } from "./db_config.mjs";

let pool;
let connection;

try {
  pool = await createPool();
  connection = await pool.getConnection();

  const who = await connection.execute(
    `SELECT
       SYS_CONTEXT('USERENV', 'SERVICE_NAME') AS service_name,
       SYS_CONTEXT('USERENV', 'SERVER_HOST') AS server_host
     FROM dual`,
  );

  const [serviceName, serverHost] = who.rows[0];
  const target =
    String(serverHost || "").toLowerCase().includes("freesql") ||
    String(serviceName || "").toLowerCase().includes("freesql")
      ? "FreeSQL"
      : "Oracle Database";

  console.log(`Connected to ${target}`);
} finally {
  if (connection) await connection.close();
  if (pool) await pool.close(0);
}
"""

_ = (RUNTIME_DIR / "connection_check.mjs").write_text(connection_check_js, encoding="utf-8")
print("Connection check ready.")

Connection check ready.


In [8]:
connection_result = subprocess.run(
    ["node", "connection_check.mjs"],
    cwd=RUNTIME_DIR,
    env=RUN_ENV,
    text=True,
    encoding="utf-8",
    errors="replace",
    capture_output=True,
)

clean_connection_output = "\n".join(
    line for line in connection_result.stdout.splitlines()
    if line.strip() and "injected env" not in line
)
if clean_connection_output:
    print(clean_connection_output)

if connection_result.returncode != 0:
    print(connection_result.stderr[-2000:])
    raise RuntimeError("Database connection check failed.")

Connected to FreeSQL


## 7. Create the Demo Embeddings

The demo embedding model keeps the notebook fully runnable without an external model key. It implements the standard LangChain.js `Embeddings` interface, so the rest of the workflow looks the same as it would with a production embedding provider.

In [9]:
embeddings_js = r'''
import { Embeddings } from "@langchain/core/embeddings";

export class DemoEmbeddings extends Embeddings {
  constructor() {
    super({});
    this.dimensions = 12;
    this.terms = [
      "oracle",
      "database",
      "vector",
      "langchain",
      "javascript",
      "typescript",
      "rag",
      "metadata",
      "freesql",
      "agent",
      "retrieval",
      "enterprise",
    ];
  }

  textToVector(text) {
    const lower = text.toLowerCase();
    const vector = this.terms.map((term) => {
      const matches = lower.match(new RegExp(term, "g"));
      return matches ? matches.length : 0;
    });

    for (const token of lower.split(/[^a-z0-9]+/).filter(Boolean)) {
      let bucket = 0;
      for (const char of token) {
        bucket = (bucket + char.charCodeAt(0)) % this.dimensions;
      }
      vector[bucket] += 0.15;
    }

    const norm = Math.sqrt(vector.reduce((sum, value) => sum + value * value, 0)) || 1;
    return vector.map((value) => Number((value / norm).toFixed(6)));
  }

  async embedQuery(text) {
    return this.textToVector(text);
  }

  async embedDocuments(texts) {
    return texts.map((text) => this.textToVector(text));
  }
}
'''

_ = (RUNTIME_DIR / "demo_embeddings.mjs").write_text(embeddings_js, encoding="utf-8")
print("Demo embedding model ready.")

Demo embedding model ready.


## 8. Create the Demo Corpus

The corpus uses short, structured documents with metadata. That lets the retrieval section demonstrate semantic search and JSON metadata filtering against the same Oracle vector table.

In [10]:
corpus_js = r'''
import { Document } from "@langchain/core/documents";
import { HumanMessage, SystemMessage } from "@langchain/core/messages";
import { ChatOpenAI } from "@langchain/openai";
import { llmApiKey, llmModel } from "./db_config.mjs";

export function docsForDemo() {
  return [
    new Document({
      pageContent:
        "Oracle AI Database stores vectors alongside operational data, so RAG applications can keep retrieval close to governed enterprise records.",
      metadata: { source: "oracle-ai-database", category: "database", product: "Oracle AI Database", priority: "high", audience: "architect" },
    }),
    new Document({
      pageContent:
        "LangChain.js RAG applications use OracleVS with Oracle AI Database as a vector store for adding documents, preserving metadata, and running similarity search retrieval.",
      metadata: { source: "langchain-js", category: "framework", product: "LangChain.js", priority: "high", audience: "developer" },
    }),
    new Document({
      pageContent:
        "FreeSQL gives developers a hosted Oracle Database schema for tutorials that only need normal schema table operations.",
      metadata: { source: "freesql", category: "setup", product: "FreeSQL", priority: "medium", audience: "developer" },
    }),
    new Document({
      pageContent:
        "Metadata filters narrow OracleVS retrieval by product, source, category, tenant, priority, or other JSON metadata fields.",
      metadata: { source: "metadata-filtering", category: "retrieval", product: "OracleVS", priority: "high", audience: "developer" },
    }),
    new Document({
      pageContent:
        "Maximal marginal relevance helps LangChain.js retrieve a diverse context set instead of returning several near-duplicate chunks.",
      metadata: { source: "mmr", category: "retrieval", product: "LangChain.js", priority: "medium", audience: "developer" },
    }),
    new Document({
      pageContent:
        "A production RAG service can swap the demo embedding class for OpenAI, OCI Generative AI, or Oracle-backed embeddings without changing OracleVS retrieval code.",
      metadata: { source: "embedding-provider", category: "architecture", product: "LangChain.js", priority: "medium", audience: "architect" },
    }),
    new Document({
      pageContent:
        "Autonomous AI Database and local Oracle Database use the same node-oracledb connection pattern when the schema can create and manage application tables.",
      metadata: { source: "connection-options", category: "setup", product: "node-oracledb", priority: "medium", audience: "developer" },
    }),
  ];
}

export function docRow(doc, distance, rank) {
  return {
    rank,
    distance: distance === undefined ? null : Number(distance.toFixed(6)),
    source: doc.metadata.source,
    product: doc.metadata.product,
    category: doc.metadata.category,
    snippet: doc.pageContent,
  };
}

export function buildContext(rows) {
  return rows
    .map((row) => `[${row.rank}] source=${row.source}; product=${row.product}; category=${row.category}; snippet=${row.snippet}`)
    .join("\n");
}

function contentToText(content) {
  if (typeof content === "string") return content;
  if (Array.isArray(content)) {
    return content
      .map((part) => {
        if (typeof part === "string") return part;
        if (part && typeof part === "object" && "text" in part) return part.text;
        return JSON.stringify(part);
      })
      .join("");
  }
  return String(content ?? "");
}

export async function buildAnswer(question, rows) {
  if (!llmApiKey) {
    throw new Error("Missing OPENAI_API_KEY or MODEL_PROVIDER_API_KEY. Set one of these values in .env to run the RAG generation step.");
  }

  const context = buildContext(rows);
  const model = new ChatOpenAI({
    apiKey: llmApiKey,
    model: llmModel,
    temperature: 0,
  });

  const response = await model.invoke([
    new SystemMessage(
      "You are a concise technical assistant. Answer only from the retrieved context. If the context is insufficient, say what is missing."
    ),
    new HumanMessage(
      `Question:\n${question}\n\nRetrieved context:\n${context}\n\nWrite a grounded answer in 3-5 sentences and mention the Oracle/LangChain components that support the answer.`
    ),
  ]);

  return [
    `Question: ${question}`,
    "",
    `Grounded answer generated by ${llmModel}:`,
    contentToText(response.content).trim(),
    "",
    "Retrieved context used:",
    context,
  ].join("\n");
}
'''

_ = (RUNTIME_DIR / "demo_corpus.mjs").write_text(corpus_js, encoding="utf-8")
print("Demo corpus ready.")

Demo corpus ready.


## 9. Build the OracleVS Setup Step

This module creates a LangChain.js `OracleVS` store from the demo documents. It also resets the demo table on rerun, checks tablespace quota, and optionally creates an HNSW vector index.

In [11]:
oracle_vs_setup_js = r"""
import {
  DistanceStrategy,
  OracleVS,
  VectorElementFormat,
  createIndex,
  dropTablePurge,
} from "@oracle/langchain-oracledb";

import { createVectorIndex, tableName, writeResults } from "./db_config.mjs";
import { checkTablespaceQuota, runTablePreflight, tableExists } from "./db_preflight.mjs";
import { docsForDemo } from "./demo_corpus.mjs";
import { DemoEmbeddings } from "./demo_embeddings.mjs";

export async function prepareVectorStore(pool, connection, summary) {
  await runTablePreflight(connection, summary);

  if (await tableExists(connection, tableName)) {
    await dropTablePurge(connection, tableName);
    summary.push({ step: "reset_vector_store", status: "PASS", detail: `Dropped existing ${tableName}` });
  } else {
    summary.push({ step: "reset_vector_store", status: "PASS", detail: `${tableName} did not exist before this run` });
  }

  const quotaCheck = await checkTablespaceQuota(connection);
  summary.push({ step: "tablespace_quota", status: quotaCheck.status, detail: quotaCheck.detail });
  if (quotaCheck.status === "MISSING") {
    writeResults({
      tableName,
      summary,
      corpus: [],
      searchResults: [],
      filteredResults: [],
      mmrResults: [],
      ragAnswer: "The database connection works, but this schema does not have enough remaining USERS tablespace quota to create and populate the OracleVS demo table.",
      integrationSurface: [],
    });
    throw new Error(quotaCheck.detail);
  }

  const embeddings = new DemoEmbeddings();
  const documents = docsForDemo();
  const vectorStore = await OracleVS.fromDocuments(documents, embeddings, {
    client: pool,
    tableName,
    query: "Oracle AI Database LangChain.js RAG",
    distanceStrategy: DistanceStrategy.COSINE,
    format: VectorElementFormat.FLOAT32,
    description: "LangChain.js RAG demo table for Oracle AI Database",
  });

  summary.push({ step: "oraclevs_ingest", status: "PASS", detail: `Inserted ${documents.length} documents into ${tableName}` });

  if (createVectorIndex) {
    try {
      const vectorIndexName = `${tableName}_HNSW_IDX`;
      await createIndex(connection, vectorStore, { idxName: vectorIndexName, idxType: "HNSW", accuracy: 90, parallel: 1 });
      summary.push({ step: "vector_index", status: "PASS", detail: `Created or reused ${vectorIndexName}` });
    } catch (error) {
      summary.push({ step: "vector_index", status: "OPTIONAL", detail: `Vector index skipped by database: ${String(error.message || error).split("\n")[0].slice(0, 160)}` });
    }
  } else {
    summary.push({ step: "vector_index", status: "READY", detail: "Set LANGCHAIN_JS_CREATE_INDEX=true to create an HNSW vector index when the schema has enough quota" });
  }

  return { vectorStore, documents };
}
"""

_ = (RUNTIME_DIR / "oracle_vs_setup.mjs").write_text(oracle_vs_setup_js, encoding="utf-8")
print("OracleVS setup step ready.")

OracleVS setup step ready.


## 10. Build the Retrieval Examples

This module shows three LangChain.js retrieval modes over the same Oracle table: similarity search, metadata-filtered search, and maximal marginal relevance.

In [12]:
retrieval_examples_js = r"""
import { buildAnswer, docRow } from "./demo_corpus.mjs";

export async function runRetrievalExamples(vectorStore, summary) {
  const question = "How can a JavaScript RAG application use OracleVS with Oracle AI Database as a vector store for retrieval?";
  const rawResults = await vectorStore.similaritySearchWithScore(question, 4);
  const searchResults = rawResults.map(([doc, score], index) => docRow(doc, score, index + 1));
  summary.push({ step: "similarity_search", status: "PASS", detail: `Retrieved ${searchResults.length} documents for the main query` });

  const filteredRawResults = await vectorStore.similaritySearchWithScore(
    "How do metadata filters improve retrieval?",
    3,
    { category: { "$eq": "retrieval" } },
  );
  const filteredResults = filteredRawResults.map(([doc, score], index) => docRow(doc, score, index + 1));
  summary.push({ step: "metadata_filter", status: "PASS", detail: `Retrieved ${filteredResults.length} retrieval-category documents` });

  const mmrDocs = await vectorStore.maxMarginalRelevanceSearch(
    "What does the JavaScript integration provide for RAG?",
    { k: 3, fetchK: 5, lambda: 0.5 },
  );
  const mmrResults = mmrDocs.map((doc, index) => docRow(doc, undefined, index + 1));
  summary.push({ step: "mmr_retrieval", status: "PASS", detail: `Selected ${mmrResults.length} diverse documents` });
  summary.push({ step: "rag_generation", status: "PASS", detail: "Generated a grounded answer from retrieved OracleVS context" });

  return {
    question,
    searchResults,
    filteredResults,
    mmrResults,
    ragAnswer: await buildAnswer(question, searchResults),
  };
}
"""

_ = (RUNTIME_DIR / "retrieval_examples.mjs").write_text(retrieval_examples_js, encoding="utf-8")
print("Retrieval examples ready.")

Retrieval examples ready.


## 11. Build the End-to-End Runner

The runner is intentionally short. It opens the database connection, calls the OracleVS setup, runs retrieval, inspects the generated table, and writes one result payload for the notebook to display.

In [13]:
workflow_js = r"""
import { createPool, tableName, writeResults } from "./db_config.mjs";
import { inspectVectorTable } from "./db_inspection.mjs";
import { docsForDemo } from "./demo_corpus.mjs";
import { prepareVectorStore } from "./oracle_vs_setup.mjs";
import { runRetrievalExamples } from "./retrieval_examples.mjs";

async function run() {
  const summary = [];
  let pool;
  let connection;

  try {
    pool = await createPool();
    connection = await pool.getConnection();
    summary.push({ step: "connect", status: "PASS", detail: "Connected with node-oracledb" });

    const { vectorStore, documents } = await prepareVectorStore(pool, connection, summary);
    const retrieval = await runRetrievalExamples(vectorStore, summary);
    const tableInspection = await inspectVectorTable(connection, tableName);

    writeResults({
      tableName,
      summary,
      tableInspection,
      corpus: docsForDemo().map((doc, index) => ({
        id: index + 1,
        source: doc.metadata.source,
        category: doc.metadata.category,
        product: doc.metadata.product,
        priority: doc.metadata.priority,
        audience: doc.metadata.audience,
        text: doc.pageContent,
      })),
      ...retrieval,
      integrationSurface: [
        { component: "OracleVS", purpose: "Vector storage and similarity search", demonstrated: "yes" },
        { component: "createIndex", purpose: "Create HNSW or IVF vector indexes when supported by the database environment", demonstrated: "opt-in" },
        { component: "DistanceStrategy", purpose: "Cosine, dot product, Euclidean, and other vector distance strategies", demonstrated: "yes" },
        { component: "VectorElementFormat", purpose: "Dense vector storage format selection", demonstrated: "yes" },
        { component: "OracleDocLoader", purpose: "Load files or database content through Oracle AI Database", demonstrated: "optional extension" },
        { component: "OracleTextSplitter", purpose: "Split text through Oracle text processing", demonstrated: "optional extension" },
        { component: "OracleEmbeddings", purpose: "Generate embeddings through Oracle AI Database", demonstrated: "optional extension" },
        { component: "OracleSummary", purpose: "Summarize text through Oracle AI Database", demonstrated: "optional extension" },
      ],
    });

    console.log(`Workflow completed: ${documents.length} documents ingested into ${tableName}; ${retrieval.searchResults.length} similarity results, ${retrieval.filteredResults.length} filtered results, ${retrieval.mmrResults.length} MMR results.`);
  } finally {
    if (connection) await connection.close();
    if (pool) await pool.close(0);
  }
}

run().catch((error) => {
  console.error(error.message || error);
  process.exitCode = 1;
});
"""

_ = (RUNTIME_DIR / "oracle_langchain_rag_demo.mjs").write_text(workflow_js, encoding="utf-8")
print("End-to-end runner ready.")

End-to-end runner ready.


## 12. Run the End-to-End Workflow

This cell runs the JavaScript application. A successful run means the notebook connected to Oracle Database, created the `OracleVS` table, inserted documents, and executed the retrieval examples.

In [14]:
run_result = subprocess.run(
    [NPM_BIN, "run", "demo"],
    cwd=RUNTIME_DIR,
    env=RUN_ENV,
    text=True,
    encoding="utf-8",
    errors="replace",
    capture_output=True,
)

if run_result.stdout.strip():
    clean_stdout = "\n".join(
        line for line in run_result.stdout.splitlines()
        if line.strip() and not line.startswith(">") and "injected env" not in line
    )
    if clean_stdout:
        print(clean_stdout[-1200:])
if run_result.stderr.strip() and run_result.returncode != 0:
    print(run_result.stderr[-3000:])

if run_result.returncode != 0:
    results_path = RUNTIME_DIR / "rag_results.json"
    if results_path.exists():
        diagnostic_payload = json.loads(results_path.read_text(encoding="utf-8"))
        quota_missing = any(
            row.get("step") == "tablespace_quota" and row.get("status") == "MISSING"
            for row in diagnostic_payload.get("summary", [])
        )
        if quota_missing:
            print("Preflight stopped before OracleVS because this schema does not have enough remaining quota.")
        else:
            raise RuntimeError("The LangChain.js Oracle RAG demo failed. Review the JavaScript error above.")
    else:
        raise RuntimeError("The LangChain.js Oracle RAG demo failed. Review the JavaScript error above.")

Workflow completed: 7 documents ingested into LC_JS_RAG_DEMO; 4 similarity results, 2 filtered results, 3 MMR results.


## 13. Inspect the Oracle Vector Store

In [15]:
import json
from pathlib import Path

import pandas as pd
try:
    from IPython.display import HTML, Markdown, display
except ImportError:
    class HTML(str):
        pass

    class Markdown(str):
        pass

    def display(value):
        print(value)


def show_table(df, caption=None):
    # Display compact tutorial tables without the pandas row index.
    if caption:
        display(Markdown(caption))
    display(HTML(df.to_html(index=False, escape=False)))

results_path = RUNTIME_DIR / "rag_results.json"
payload = json.loads(results_path.read_text(encoding="utf-8"))

display(Markdown(
    f"**Vector table:** `{payload['tableName']}`\n\n"
    "The status table shows each database and LangChain.js capability checked by the notebook."
))
show_table(pd.DataFrame(payload["summary"]))
preflight_stopped = not payload.get("corpus")

**Vector table:** `LC_JS_RAG_DEMO`

The status table shows each database and LangChain.js capability checked by the notebook.

step,status,detail
connect,PASS,Connected with node-oracledb
table_operations,PASS,"Created, inserted, selected 1 row, and dropped a test table"
reset_vector_store,PASS,Dropped existing LC_JS_RAG_DEMO
tablespace_quota,PASS,USERS: 2.00 MB used of 10.00 MB; 8.00 MB remains
oraclevs_ingest,PASS,Inserted 7 documents into LC_JS_RAG_DEMO
vector_index,READY,Set LANGCHAIN_JS_CREATE_INDEX=true to create an HNSW vector index when the schema has enough quota
similarity_search,PASS,Retrieved 4 documents for the main query
metadata_filter,PASS,Retrieved 2 retrieval-category documents
mmr_retrieval,PASS,Selected 3 diverse documents
rag_generation,PASS,Generated a grounded answer from retrieved OracleVS context


In [16]:
if preflight_stopped:
    display(Markdown("The database connection works, but the vector table was not created because the preflight stopped first."))
else:
    inspection = payload["tableInspection"]
    display(Markdown(
        f"**What OracleVS created**\n\n"
        f"`OracleVS` created one Oracle table, stored `{inspection['rowCount']}` LangChain documents, "
        f"and used about `{inspection['storageMb']} MB` of table segment space."
    ))
    show_table(
        pd.DataFrame(inspection["columns"]),
        "The generated table keeps document text, JSON metadata, and the native Oracle `VECTOR` column together."
    )
    user_indexes = [
        row for row in inspection["indexes"]
        if not str(row["index"]).startswith("SYS_")
    ]
    if user_indexes:
        display(Markdown("**User-created vector indexes**"))
        show_table(pd.DataFrame(user_indexes))
    else:
        display(Markdown("No user-created vector index is shown because `LANGCHAIN_JS_CREATE_INDEX` is `false` by default for FreeSQL-friendly runs."))

**What OracleVS created**

`OracleVS` created one Oracle table, stored `7` LangChain documents, and used about `0.063 MB` of table segment space.

The generated table keeps document text, JSON metadata, and the native Oracle `VECTOR` column together.

column,type
ID,RAW
EXTERNAL_ID,VARCHAR2
EMBEDDING,VECTOR
TEXT,CLOB
METADATA,JSON


No user-created vector index is shown because `LANGCHAIN_JS_CREATE_INDEX` is `false` by default for FreeSQL-friendly runs.

## 14. Review the Demo Corpus

In [17]:
corpus_df = pd.DataFrame(payload["corpus"])
if preflight_stopped:
    display(Markdown("Document ingestion did not run because the database preflight stopped first."))
else:
    show_table(
        corpus_df.reindex(columns=["id", "category", "product", "priority", "audience", "text"]),
        "This small corpus is intentionally structured with metadata so the retrieval section can show both semantic search and metadata filtering."
    )

This small corpus is intentionally structured with metadata so the retrieval section can show both semantic search and metadata filtering.

id,category,product,priority,audience,text
1,database,Oracle AI Database,high,architect,"Oracle AI Database stores vectors alongside operational data, so RAG applications can keep retrieval close to governed enterprise records."
2,framework,LangChain.js,high,developer,"LangChain.js RAG applications use OracleVS with Oracle AI Database as a vector store for adding documents, preserving metadata, and running similarity search retrieval."
3,setup,FreeSQL,medium,developer,FreeSQL gives developers a hosted Oracle Database schema for tutorials that only need normal schema table operations.
4,retrieval,OracleVS,high,developer,"Metadata filters narrow OracleVS retrieval by product, source, category, tenant, priority, or other JSON metadata fields."
5,retrieval,LangChain.js,medium,developer,Maximal marginal relevance helps LangChain.js retrieve a diverse context set instead of returning several near-duplicate chunks.
6,architecture,LangChain.js,medium,architect,"A production RAG service can swap the demo embedding class for OpenAI, OCI Generative AI, or Oracle-backed embeddings without changing OracleVS retrieval code."
7,setup,node-oracledb,medium,developer,Autonomous AI Database and local Oracle Database use the same node-oracledb connection pattern when the schema can create and manage application tables.


## 15. Compare Retrieval Modes

In [18]:
search_df = pd.DataFrame(payload["searchResults"])
filtered_df = pd.DataFrame(payload["filteredResults"])
mmr_df = pd.DataFrame(payload["mmrResults"])

if preflight_stopped:
    display(Markdown("Retrieval examples are skipped because document ingestion did not run."))
else:
    show_table(
        search_df.reindex(columns=["rank", "distance", "source", "category", "product", "snippet"]),
        "**Similarity search** retrieves the nearest documents by vector distance. Lower cosine distance is closer."
    )
    show_table(
        filtered_df.reindex(columns=["rank", "distance", "source", "category", "product", "snippet"]),
        "**Metadata-filtered search** applies a JSON metadata predicate before returning context. Here the filter is `category = retrieval`."
    )
    show_table(
        mmr_df.reindex(columns=["rank", "source", "category", "product", "snippet"]),
        "**MMR retrieval** balances relevance and diversity, so the context set does not collapse into near-duplicate chunks."
    )

**Similarity search** retrieves the nearest documents by vector distance. Lower cosine distance is closer.

rank,distance,source,category,product,snippet
1,0.123145,oracle-ai-database,database,Oracle AI Database,"Oracle AI Database stores vectors alongside operational data, so RAG applications can keep retrieval close to governed enterprise records."
2,0.148912,embedding-provider,architecture,LangChain.js,"A production RAG service can swap the demo embedding class for OpenAI, OCI Generative AI, or Oracle-backed embeddings without changing OracleVS retrieval code."
3,0.153764,langchain-js,framework,LangChain.js,"LangChain.js RAG applications use OracleVS with Oracle AI Database as a vector store for adding documents, preserving metadata, and running similarity search retrieval."
4,0.172201,connection-options,setup,node-oracledb,Autonomous AI Database and local Oracle Database use the same node-oracledb connection pattern when the schema can create and manage application tables.


**Metadata-filtered search** applies a JSON metadata predicate before returning context. Here the filter is `category = retrieval`.

rank,distance,source,category,product,snippet
1,0.092271,metadata-filtering,retrieval,OracleVS,"Metadata filters narrow OracleVS retrieval by product, source, category, tenant, priority, or other JSON metadata fields."
2,0.729846,mmr,retrieval,LangChain.js,Maximal marginal relevance helps LangChain.js retrieve a diverse context set instead of returning several near-duplicate chunks.


**MMR retrieval** balances relevance and diversity, so the context set does not collapse into near-duplicate chunks.

rank,source,category,product,snippet
1,oracle-ai-database,database,Oracle AI Database,"Oracle AI Database stores vectors alongside operational data, so RAG applications can keep retrieval close to governed enterprise records."
2,mmr,retrieval,LangChain.js,Maximal marginal relevance helps LangChain.js retrieve a diverse context set instead of returning several near-duplicate chunks.
3,embedding-provider,architecture,LangChain.js,"A production RAG service can swap the demo embedding class for OpenAI, OCI Generative AI, or Oracle-backed embeddings without changing OracleVS retrieval code."


In [19]:
if preflight_stopped:
    display(Markdown("Retrieval comparison is skipped because document ingestion did not run."))
else:
    comparison_rows = []
    for row in search_df.to_dict("records"):
        comparison_rows.append({
            "mode": "similarity",
            "rank": row["rank"],
            "source": row["source"],
            "category": row["category"],
            "distance": row.get("distance", row.get("score")),
        })
    for row in filtered_df.to_dict("records"):
        comparison_rows.append({
            "mode": "metadata filter",
            "rank": row["rank"],
            "source": row["source"],
            "category": row["category"],
            "distance": row.get("distance", row.get("score")),
        })
    for row in mmr_df.to_dict("records"):
        comparison_rows.append({
            "mode": "MMR diversity",
            "rank": row["rank"],
            "source": row["source"],
            "category": row["category"],
            "distance": "not returned",
        })
    show_table(
        pd.DataFrame(comparison_rows),
        "This comparison shows how the same vector store supports different retrieval modes through LangChain.js."
    )

This comparison shows how the same vector store supports different retrieval modes through LangChain.js.

mode,rank,source,category,distance
similarity,1,oracle-ai-database,database,0.123145
similarity,2,embedding-provider,architecture,0.148912
similarity,3,langchain-js,framework,0.153764
similarity,4,connection-options,setup,0.172201
metadata filter,1,metadata-filtering,retrieval,0.092271
metadata filter,2,mmr,retrieval,0.729846
MMR diversity,1,oracle-ai-database,database,not returned
MMR diversity,2,mmr,retrieval,not returned
MMR diversity,3,embedding-provider,architecture,not returned


## 16. Generate a Grounded Answer

In [20]:
display(Markdown(
    "The final answer below is generated by the configured chat model from the retrieved rows above. "
    "The prompt instructs the model to answer only from retrieved context.\n\n"
    "```text\n" + payload["ragAnswer"] + "\n```"
))

The final answer below is generated by the configured chat model from the retrieved rows above. The prompt instructs the model to answer only from retrieved context.

```text
Question: How can a JavaScript RAG application use OracleVS with Oracle AI Database as a vector store for retrieval?

Grounded answer generated by gpt-4o-mini:
A JavaScript RAG application can utilize OracleVS with Oracle AI Database as a vector store by leveraging the capabilities of LangChain.js. This framework allows the application to add documents, preserve metadata, and perform similarity search retrieval using the Oracle AI Database, which stores vectors alongside operational data. Additionally, the application can easily swap embedding classes, such as OpenAI or OCI Generative AI, without altering the retrieval code for OracleVS, ensuring flexibility in embedding options. This integration keeps retrieval processes close to governed enterprise records, enhancing data management and accessibility.

Retrieved context used:
[1] source=oracle-ai-database; product=Oracle AI Database; category=database; snippet=Oracle AI Database stores vectors alongside operational data, so RAG applications can keep retrieval close to governed enterprise records.
[2] source=embedding-provider; product=LangChain.js; category=architecture; snippet=A production RAG service can swap the demo embedding class for OpenAI, OCI Generative AI, or Oracle-backed embeddings without changing OracleVS retrieval code.
[3] source=langchain-js; product=LangChain.js; category=framework; snippet=LangChain.js RAG applications use OracleVS with Oracle AI Database as a vector store for adding documents, preserving metadata, and running similarity search retrieval.
[4] source=connection-options; product=node-oracledb; category=setup; snippet=Autonomous AI Database and local Oracle Database use the same node-oracledb connection pattern when the schema can create and manage application tables.
```

## TypeScript Notes

The notebook runs JavaScript because Node.js can execute `.mjs` files directly. In a TypeScript application, use the same package and imports, then compile with your normal TypeScript build.

```typescript
import {
  OracleVS,
  DistanceStrategy,
  VectorElementFormat,
  createIndex,
} from "@oracle/langchain-oracledb";
```

The integration also exposes optional Oracle AI Database utilities for document loading, text splitting, embeddings, and summarization:

```typescript
import {
  OracleDocLoader,
  OracleTextSplitter,
  OracleEmbeddings,
  OracleSummary,
} from "@oracle/langchain-oracledb";
```

## Integration Surface

This tutorial focuses on the most direct LangChain.js RAG path: `OracleVS` as the vector store, Node.js as the application runtime, and Oracle AI Database as the storage and retrieval layer.

| Component | Purpose | Covered in this tutorial |
| --- | --- | --- |
| `OracleVS` | Store and query document vectors in Oracle AI Database | Runnable end-to-end |
| `DistanceStrategy` | Choose vector distance behavior such as cosine distance | Runnable end-to-end |
| `VectorElementFormat` | Control dense vector storage format | Runnable end-to-end |
| Metadata filters | Restrict retrieval through JSON metadata predicates | Runnable end-to-end |
| MMR retrieval | Return a diverse context set instead of near-duplicate chunks | Runnable end-to-end |
| `createIndex` | Create HNSW or IVF vector indexes when the database environment supports it | Opt-in runtime path |
| `OracleDocLoader` | Load supported documents through Oracle AI Database | Extension path |
| `OracleTextSplitter` | Split text through Oracle text processing | Extension path |
| `OracleEmbeddings` | Generate embeddings through Oracle AI Database | Extension path |
| `OracleSummary` | Summarize text through Oracle AI Database | Extension path |

The runnable path is intentionally small enough to work as a first tutorial. The extension path is where a production application can add database-backed document loading, Oracle text splitting, in-database embedding generation, summarization, and vector-index tuning.


## Learn More

- [Oracle LangChain JavaScript integration guide](https://docs.oracle.com/en/database/oracle/oracle-database/26/aintg/langchain-oracledb-integration-guide/langchain-javascript.html)
- [Oracle AI Database integrations overview](https://docs.oracle.com/en/database/oracle/oracle-database/26/aintg/integrations.html)
- [LangChain.js Oracle AI Database vector store docs](https://docs.langchain.com/oss/javascript/integrations/vectorstores/oracleai)
- [npm: @oracle/langchain-oracledb](https://www.npmjs.com/package/@oracle/langchain-oracledb)
- [Source: oracle/langchain-oracle](https://github.com/oracle/langchain-oracle/tree/main/libs/js/langchain-oracledb)

## What This Proves

This notebook validates the first-party tutorial path for the LangChain TypeScript/JavaScript integration:

- `@oracle/langchain-oracledb` can create and populate an Oracle AI Database vector table from a JavaScript application.
- `OracleVS` supports document storage, semantic retrieval, metadata filtering, vector-index setup, and MMR retrieval.
- FreeSQL can be used as the hosted Oracle Database target when the schema can create normal application tables and has enough quota.
- The retrieved OracleVS context is passed to a chat model to generate the final grounded RAG answer.
- The same pattern can move to TypeScript, production embeddings, Autonomous AI Database, or local Oracle Database without changing the core LangChain RAG flow.

For continuous improvement, the next useful tutorials would be deeper ingestion examples with `OracleDocLoader` and `OracleTextSplitter`, plus production embedding and summarization examples with `OracleEmbeddings` and `OracleSummary`.
